In [1]:
import sys
import os
from pathlib import Path

# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

# RAGAS Benchmark Generator v2 - generate_with_chunks()

Phiên bản này giải quyết vấn đề treo tại HeadlinesExtractor bằng cách:
1. Chunk documents trước (pre-chunked)
2. Dùng generate_with_chunks() thay vì generate_with_langchain_docs()
3. Bỏ qua default transform pipeline nặng

Theo tài liệu RAGAS 0.4.3: https://docs.ragas.io/en/latest/howtos/customizations/testgenerator/prechunked_data/

In [2]:
import json
import time
from pathlib import Path
from typing import List

from langchain_core.documents import Document
from openai import OpenAI

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator

from configs.GetConfig import config
from configs.setting import settings
from src.b_indexing.b0_vector_db import ChromaVectorDatabase
from src.a_ingestion.a4_chunker import ChunkingDocuments

c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\instructor\providers\gemini\client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 1) Load raw docs

In [46]:
def load_documents(limit_products=100, limit_policies=50) -> List[Document]:
    documents = []
    db = ChromaVectorDatabase()

    print("📦 Loading products from ChromaDB...")
    products_data = db.get_all_documents("products_collection", limit=limit_products)
    if products_data and products_data.get("documents"):
        for doc, meta in zip(products_data["documents"], products_data["metadatas"]):
            documents.append(Document(
                page_content=doc,
                metadata={"type": "product", **(meta or {})}
            ))
        print(f"   ✅ Loaded {len(products_data['documents'])} products")
    else:
        print("   ⚠️ No products found in collection")

    print("📜 Loading policies from ChromaDB...")
    policies_data = db.get_all_documents("policies_collection", limit=limit_policies)
    if policies_data and policies_data.get("documents"):
        for doc, meta in zip(policies_data["documents"], policies_data["metadatas"]):
            documents.append(Document(
                page_content=doc,
                metadata={"type": "policy", **(meta or {})}
            ))
        print(f"   ✅ Loaded {len(policies_data['documents'])} policies")
    else:
        print("   ⚠️ No policies found in collection")

    print(f"📚 Total documents loaded: {len(documents)}")
    return documents

## 2) Chunk raw docs first (sử dụng ChunkingDocuments có sẵn)

In [47]:
def to_prechunked_documents(
    documents: List[Document],
    chunk_size: int = 1800,
    overlap: int = 200,
) -> List[Document]:
    """
    Chunk documents với logic:
    - Products: giữ nguyên (không chunk) để tránh mất thông tin tên sản phẩm
    - Policies: chunk theo config.chunking
    
    Dùng config.chunking từ config.yaml có sẵn.
    Clean surrogate characters để tránh UnicodeEncodeError.
    """
    import re
    import unicodedata
    
    chunks = []
    
    # Dùng config.chunking từ config.yaml có sẵn
    chunking_config = config.chunking
    chunker = ChunkingDocuments(chunking_config)
    splitter = chunker.build_splitter()
    
    def clean_text(text: str) -> str:
        """Clean surrogate characters và encoding issues"""
        if not text:
            return text
        # Normalize Unicode form
        text = unicodedata.normalize('NFKC', text)
        # Remove surrogate characters using regex
        text = re.sub(r'[\ud800-\udfff]', '', text)
        # Remove other problematic characters
        text = re.sub(r'[\u0000-\u0008\u000b\u000c\u000e-\u001f]', '', text)
        return text
    
    for doc_idx, doc in enumerate(documents):
        content = clean_text((doc.page_content or "").strip())
        if not content:
            continue
        
        base_meta = dict(doc.metadata or {})
        base_meta.setdefault("source_doc_index", doc_idx)
        
        doc_type = base_meta.get("type", "unknown")
        
        # Products: giữ nguyên (không chunk)
        if doc_type == "product":
            meta = dict(base_meta)
            meta["chunk_index"] = 0
            chunks.append(Document(page_content=content, metadata=meta))
            print(f"   ✅ Product {doc_idx}: giữ nguyên ({len(content)} chars)")
        
        # Policies: chunk theo config
        elif doc_type == "policy":
            parts = splitter.split_text(content)
            for chunk_idx, part in enumerate(parts):
                meta = dict(base_meta)
                meta["chunk_index"] = chunk_idx
                chunks.append(Document(page_content=part, metadata=meta))
            print(f"   ✅ Policy {doc_idx}: chia thành {len(parts)} chunks")
        
        # Others: chunk theo config (default)
        else:
            parts = splitter.split_text(content)
            for chunk_idx, part in enumerate(parts):
                meta = dict(base_meta)
                meta["chunk_index"] = chunk_idx
                chunks.append(Document(page_content=part, metadata=meta))
            print(f"   ✅ Other {doc_idx}: chia thành {len(parts)} chunks")
    
    return chunks

## 3) Setup generator

In [ ]:
# Custom embedding wrapper cho RAGAS dùng requests.post (tương thích OpenRouter)
import requests
import json
import asyncio
from typing import List
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

class OpenRouterEmbedding:
    """Custom embedding wrapper cho RAGAS dùng requests.post với API key rotation"""
    
    def __init__(self, api_keys: List[str], model: str):
        self.api_keys = api_keys
        self.model = model
        self.url = "https://openrouter.ai/api/v1/embeddings"
        self.current_key_index = 0
    
    def get_next_api_key(self):
        """Rotate to next API key"""
        self.current_key_index = (self.current_key_index + 1) % len(self.api_keys)
        return self.api_keys[self.current_key_index]
    
    @retry(
        stop=stop_after_attempt(5),  # Tăng số lần retry để có cơ hội thử nhiều keys
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type((requests.exceptions.RequestException, ValueError)),
    )
    def embed_text(self, text: str) -> List[float]:
        """Sync embedding method với retry và API key rotation"""
        # Try current key first
        api_key = self.api_keys[self.current_key_index]
        
        payload = {
            "model": self.model,
            "input": text,
            "encoding_format": "float"
        }
        
        response = requests.post(
            url=self.url,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            data=json.dumps(payload)
        )
        
        result = response.json()
        if "error" in result:
            error_msg = result["error"].lower()
            status_code = str(response.status_code)
            
            # Nếu là rate limit (429), rotate sang key khác và raise để retry
            if "rate limit" in error_msg or "429" in status_code:
                print(f"⚠️  Rate limit detected for OpenRouter key {self.current_key_index + 1}/{len(self.api_keys)}")
                next_key = self.get_next_api_key()
                print(f"🔄 Rotating to OpenRouter key {self.current_key_index + 1}/{len(self.api_keys)}")
                raise ValueError(f"Rate limit: {result['error']}")
            
            raise ValueError(f"OpenRouter API error: {result['error']}")
        
        return result["data"][0]["embedding"]
    
    async def aembed_text(self, text: str) -> List[float]:
        """Async embedding method cho RAGAS"""
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.embed_text, text)

def setup_generator():
    print("=" * 60)
    print("SETUP GENERATOR WITH API KEY ROTATION")
    print("=" * 60)

    # Dùng Gemini thay vì Groq để tránh rate limit
    model_name = config.llm.google.available[0]  # gemini-3.5-flash-lite
    print(f"LLM Model: {model_name}")
    
    # Multiple Gemini API keys (add more to .env as GEMINI_API_KEY_2, GEMINI_API_KEY_3, etc.)
    gemini_keys = [settings.GEMINI_API_KEY]
    for i in range(2, 10):  # Check for up to 9 additional keys
        key_attr = f"GEMINI_API_KEY_{i}"
        if hasattr(settings, key_attr):
            key = getattr(settings, key_attr)
            if key and "your-gemini" not in key.lower():
                gemini_keys.append(key)
    
    print(f"Gemini API Keys: {len(gemini_keys)} key(s) configured")
    for i, key in enumerate(gemini_keys):
        print(f"  Key {i+1}: {key[:10]}..." if key else "  Key {i+1}: None")

    # Tạo Gemini client với key đầu tiên
    from google import genai
    gemini_client = genai.Client(api_key=gemini_keys[0])
    generator_llm = llm_factory(model_name, provider="google", client=gemini_client)
    print("Gemini LLM setup complete")

    embed_model = getattr(config.embedding, "active", None) or "nvidia/llama-nemotron-embed-vl-1b-v2:free"
    print(f"Embedding Model: {embed_model}")
    
    # Multiple OpenRouter API keys (add more to .env as OPENROUTER_API_KEY_2, etc.)
    openrouter_keys = [settings.OPENROUTER_API_KEY]
    for i in range(2, 10):  # Check for up to 9 additional keys
        key_attr = f"OPENROUTER_API_KEY_{i}"
        if hasattr(settings, key_attr):
            key = getattr(settings, key_attr)
            if key and "your-openrouter" not in key.lower():
                openrouter_keys.append(key)
    
    print(f"OpenRouter API Keys: {len(openrouter_keys)} key(s) configured")
    for i, key in enumerate(openrouter_keys):
        print(f"  Key {i+1}: {key[:10]}..." if key else "  Key {i+1}: None")

    # Dùng custom embedding wrapper với multiple keys
    generator_embeddings = OpenRouterEmbedding(
        api_keys=openrouter_keys,
        model=embed_model
    )

    generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
    print("TestsetGenerator created")

    return generator

## 4) Generate testset

In [ ]:
def generate_testset(documents: List[Document], output_path: str, num_samples: int = 20):
    print("=" * 60)
    print("START GENERATING TEST SET (v2 - batch processing with API key rotation)")
    print("=" * 60)
    print(f"Total documents: {len(documents)}")
    print(f"Total questions needed: {num_samples}")
    
    # Batch processing config
    batch_docs = 10  # 10 documents mỗi batch
    batch_questions = 5  # 5 câu hỏi mỗi batch (tỷ lệ 1:2)
    delay_between_batches = 60  # 60 giây delay giữa các batch để tránh rate limit
    
    # Collect multiple Gemini API keys
    gemini_keys = [settings.GEMINI_API_KEY]
    for i in range(2, 10):
        key_attr = f"GEMINI_API_KEY_{i}"
        if hasattr(settings, key_attr):
            key = getattr(settings, key_attr)
            if key and "your-gemini" not in key.lower():
                gemini_keys.append(key)
    
    print(f"Gemini API Keys available: {len(gemini_keys)}")
    
    generator = setup_generator()

    # Chunk với logic mới: products giữ nguyên, chỉ chunk policies
    all_chunks = to_prechunked_documents(documents, chunk_size=1800, overlap=200)
    print(f"Total chunks after chunking: {len(all_chunks)}")
    
    # Pipeline tối giản
    from ragas.testset.persona import Persona
    from ragas.testset.transforms import Parallel, OverlapScoreBuilder
    from ragas.testset.transforms.extractors.llm_based import NERExtractor, ThemesExtractor
    from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
    
    is_chunk = lambda n: n.type.name == "CHUNK"
    
    transforms = [
        Parallel(
            ThemesExtractor(llm=generator.llm, filter_nodes=is_chunk),
            NERExtractor(llm=generator.llm, filter_nodes=is_chunk)
        ),
        OverlapScoreBuilder(threshold=0.3, filter_nodes=is_chunk),
    ]
    
    personas = [
        Persona(name="Khách mua điện thoại", role_description="Quan tâm giá, pin, camera, khuyến mãi."),
        Persona(name="Khách mua laptop", role_description="Quan tâm cấu hình, RAM, CPU, bảo hành."),
        Persona(name="Khách hỏi chính sách", role_description="Quan tâm đổi trả, giao hàng, bảo hành."),
    ]
    
    query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator.llm, property_name="entities"), 1.0)
    ]
    
    run_config = RunConfig(max_workers=1, max_retries=3, max_wait=60)
    
    # Batch processing
    all_formatted_data = []
    total_batches = (num_samples + batch_questions - 1) // batch_questions
    current_chunk_index = 0
    current_gemini_key_index = 0
    
    print(f"\nBatch processing config:")
    print(f"   - {batch_docs} documents per batch")
    print(f"   - {batch_questions} questions per batch")
    print(f"   - {delay_between_batches}s delay between batches")
    print(f"   - Total batches: {total_batches}")
    print("-" * 60)
    
    for batch_num in range(total_batches):
        print(f"\nBATCH {batch_num + 1}/{total_batches}")
        print("=" * 60)
        
        # Lấy chunks cho batch này
        batch_chunks = all_chunks[current_chunk_index:current_chunk_index + batch_docs]
        current_chunk_index += batch_docs
        
        if not batch_chunks:
            print("WARNING: No chunks left, stopping batch processing")
            break
        
        print(f"Chunks in this batch: {len(batch_chunks)}")
        print(f"Questions to generate: {batch_questions}")
        print(f"Using Gemini key {current_gemini_key_index + 1}/{len(gemini_keys)}")
        
        # Sinh testset cho batch này với retry logic
        max_retries_per_key = 2
        retry_count = 0
        
        while retry_count < max_retries_per_key * len(gemini_keys):
            try:
                start_time = time.time()
                testset = generator.generate_with_chunks(
                    chunks=batch_chunks,
                    testset_size=batch_questions,
                    run_config=run_config,
                    transforms=transforms,
                    query_distribution=query_distribution,
                    with_debugging_logs=True,
                    raise_exceptions=True,
                )
                elapsed_time = time.time() - start_time
                
                print(f"Batch {batch_num + 1} time: {elapsed_time:.1f}s")
                print("Batch completed!")
                
                # Convert testset to formatted data
                testset_df = testset.to_pandas()
                for idx, row in testset_df.iterrows():
                    all_formatted_data.append({
                        "id": f"ragas_batch{batch_num + 1}_{idx}",
                        "category": "ragas_generated",
                        "question": row["user_input"],
                        "ground_truth": row["reference"],
                        "context": row["reference_contexts"],
                        "synthesizer_type": row["synthesizer_name"],
                    })
                
                print(f"Generated {len(testset_df)} questions in this batch")
                print(f"Total: {len(all_formatted_data)}/{num_samples} questions")
                
                # Break retry loop on success
                break
                
            except Exception as e:
                error_str = str(e).lower()
                if "429" in error_str or "resource_exhausted" in error_str or "rate limit" in error_str:
                    print(f"Rate limit hit on Gemini key {current_gemini_key_index + 1}")
                    retry_count += 1
                    
                    # Rotate to next key
                    if current_gemini_key_index < len(gemini_keys) - 1:
                        current_gemini_key_index += 1
                        print(f"Rotating to Gemini key {current_gemini_key_index + 1}/{len(gemini_keys)}")
                        
                        # Re-create generator with new key
                        from google import genai
                        new_gemini_client = genai.Client(api_key=gemini_keys[current_gemini_key_index])
                        new_generator_llm = llm_factory(model_name, provider="google", client=new_gemini_client)
                        
                        # Get OpenRouter embedding from current generator
                        generator_embeddings = generator.embedding_model
                        
                        # Create new generator
                        generator = TestsetGenerator(llm=new_generator_llm, embedding_model=generator_embeddings)
                        
                        # Update transforms with new LLM
                        transforms = [
                            Parallel(
                                ThemesExtractor(llm=generator.llm, filter_nodes=is_chunk),
                                NERExtractor(llm=generator.llm, filter_nodes=is_chunk)
                            ),
                            OverlapScoreBuilder(threshold=0.3, filter_nodes=is_chunk),
                        ]
                        
                        query_distribution = [
                            (SingleHopSpecificQuerySynthesizer(llm=generator.llm, property_name="entities"), 1.0)
                        ]
                        
                        # Wait a bit before retry
                        time.sleep(5)
                        continue
                    else:
                        print("All Gemini keys exhausted, waiting 60s before retry...")
                        time.sleep(60)
                        current_gemini_key_index = 0  # Reset to first key
                        continue
                else:
                    # Non-rate-limit error, raise immediately
                    raise
        
        # Delay giữa các batch để tránh rate limit (trừ batch cuối)
        if batch_num < total_batches - 1 and len(all_formatted_data) < num_samples:
            print(f"\nWaiting {delay_between_batches}s to avoid rate limit...")
            time.sleep(delay_between_batches)
    
    # Save all formatted data
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for item in all_formatted_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    
    print("\n" + "=" * 60)
    print(f"DONE! Generated {len(all_formatted_data)} questions to {output_path}")
    print("=" * 60)
    
    return all_formatted_data

## 5) Run pipeline

In [50]:
# Load documents
documents = load_documents()

📦 Loading products from ChromaDB...
   ✅ Loaded 100 products
📜 Loading policies from ChromaDB...
   ✅ Loaded 3 policies
📚 Total documents loaded: 103


In [51]:
# Xem 1-2 documents mẫu
print("📄 Sample documents:")
print("-" * 60)
for i in range(min(2, len(documents))):
    print(f"\n--- Document {i+1} ---")
    print(f"Type: {documents[i].metadata.get('type', 'unknown')}")
    print(f"Source: {documents[i].metadata.get('source', 'unknown')}")
    print(f"Content preview: {documents[i].page_content[:200]}...")
    print(f"Full content length: {len(documents[i].page_content)} chars")

📄 Sample documents:
------------------------------------------------------------

--- Document 1 ---
Type: product
Source: unknown
Content preview: Sản phẩm: Nubia Neo 5 GT Special Edition 12GB 256GB

Thương hiệu: Nubia | Danh mục: phone

Thông tin giá & Kho hàng:
- Giá thực tế: 13,990,000 VNĐ
- Giá gốc niêm yết: 13,990,000 VNĐ
- Mức giảm giá: Kh...
Full content length: 786 chars

--- Document 2 ---
Type: product
Source: unknown
Content preview: Sản phẩm: Laptop Lenovo Yoga Slim 7 14IPH11 83QM0076VN

Thương hiệu: Lenovo | Danh mục: laptop

Thông tin giá & Kho hàng:
- Giá thực tế: 40,990,000 VNĐ
- Giá gốc niêm yết: 42,990,000 VNĐ
- Mức giảm gi...
Full content length: 2583 chars


In [52]:
# Test OpenRouter Embedding API trực tiếp (dùng requests.post giống notebook 02)
import requests
import json
from configs.setting import settings
from configs.GetConfig import config

sample_chunk_text = "Test text for embedding"

api_key = settings.OPENROUTER_API_KEY
if not api_key or "your-openrouter" in api_key:
    raise ValueError("Vui lòng điền OPENROUTER_API_KEY thật vào file .env!")

payload = {
    "model": config.embedding.active,
    "input": sample_chunk_text,
    "encoding_format": "float"
}

response = requests.post(
    url="https://openrouter.ai/api/v1/embeddings",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    data=json.dumps(payload)
)

result = response.json()
print(json.dumps(result, indent=2))

if "error" in result:
    print("Lỗi từ OpenRouter API:", result["error"])
else:
    embedding = result["data"][0]["embedding"]
    print(f"✅ Nhúng thành công! Số chiều của Vector (Dimension): {len(embedding)}")
    print("5 giá trị đầu tiên của Vector:", embedding[:5])

{
  "object": "list",
  "data": [
    {
      "object": "embedding",
      "embedding": [
        0.00853729248046875,
        -0.003570556640625,
        0.0243072509765625,
        -0.0063018798828125,
        0.0183868408203125,
        -0.0202789306640625,
        0.01032257080078125,
        -0.031341552734375,
        -0.0174407958984375,
        -0.056549072265625,
        -0.0270843505859375,
        0.01490020751953125,
        0.007030487060546875,
        0.0003750324249267578,
        0.0009284019470214844,
        0.0211639404296875,
        -0.004749298095703125,
        0.0288543701171875,
        -0.036468505859375,
        0.002750396728515625,
        0.03082275390625,
        -0.021728515625,
        0.003143310546875,
        0.020233154296875,
        -0.0017261505126953125,
        0.0163116455078125,
        0.01983642578125,
        0.0276031494140625,
        -0.047271728515625,
        -0.0295562744140625,
        -0.00859832763671875,
        0.01670837402343

In [53]:
# Generate testset
testset = generate_testset(
    documents, 
    "src/h_evaluation/test_sets/ragas_generated_v2.jsonl", 
    num_samples=1
)

START GENERATING TEST SET (v2 - batch processing)
Total documents: 103
Total questions needed: 1
⚙️  SETUP GENERATOR
🤖 LLM Model: gemini-3.5-flash-lite
🔑 Gemini API Key: AQ.Ab8RN6J...
✅ Gemini LLM setup complete
🔤 Embedding Model: nvidia/llama-nemotron-embed-vl-1b-v2:free
✅ TestsetGenerator created
   ✅ Product 0: giữ nguyên (786 chars)
   ✅ Product 1: giữ nguyên (2583 chars)
   ✅ Product 2: giữ nguyên (3625 chars)
   ✅ Product 3: giữ nguyên (3613 chars)
   ✅ Product 4: giữ nguyên (3681 chars)
   ✅ Product 5: giữ nguyên (3388 chars)
   ✅ Product 6: giữ nguyên (802 chars)
   ✅ Product 7: giữ nguyên (2404 chars)
   ✅ Product 8: giữ nguyên (2414 chars)
   ✅ Product 9: giữ nguyên (660 chars)
   ✅ Product 10: giữ nguyên (2161 chars)
   ✅ Product 11: giữ nguyên (2160 chars)
   ✅ Product 12: giữ nguyên (2591 chars)
   ✅ Product 13: giữ nguyên (2593 chars)
   ✅ Product 14: giữ nguyên (3159 chars)
   ✅ Product 15: giữ nguyên (3235 chars)
   ✅ Product 16: giữ nguyên (3143 chars)
   ✅ Product 17:

Applying ThemesExtractor:   0%|          | 0/10 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/10 [00:00<?, ?it/s]

Task failed with InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 47.324341655s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerP

InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 47.324341655s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fr

In [ ]:
# Xem tất cả các thư viện và phiên bản hiện tại trong môi trường
import subprocess
import sys

print("=" * 60)
print("📦 DANH SÁCH THƯ VIỆN VÀ PHIÊN BẢN (pip freeze)")
print("=" * 60)

try:
    result = subprocess.run([sys.executable, "-m", "pip", "freeze"], 
                          capture_output=True, text=True, check=True)
    packages = result.stdout.strip().split('\n')
    
    # Sắp xếp theo tên thư viện
    packages.sort()
    
    for pkg in packages:
        print(pkg)
        
    print("=" * 60)
    print(f"📊 Tổng số thư viện: {len(packages)}")
    print("=" * 60)
except subprocess.CalledProcessError as e:
    print(f"❌ Lỗi khi chạy pip freeze: {e}")